# 02. The CPython memory model

## Learning goals
1. Distinguish identity (`is`, `id`) from equality (`==`).
2. Read reference counts and explain when an object dies.
3. Explain interning and the small-integer cache.
4. Show how the cyclic garbage collector frees cycles.
5. Measure object sizes with `sys.getsizeof` and read bytecode with `dis`.

In [ ]:
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent):
    if (_p / "src" / "understanding_python").exists():
        sys.path.insert(0, str(_p / "src"))
print("python", sys.version.split()[0])

## Names are references
An assignment binds a name to an object. It never copies. `id(x)` is the object address in CPython.

In [ ]:
a = [1, 2, 3]
b = a
c = [1, 2, 3]
assert a is b and a == c and a is not c
assert id(a) == id(b) != id(c)
b.append(4)
assert a == [1, 2, 3, 4]

## Reference counting
Each object stores a count of references. When it reaches zero, CPython frees the object at once.
`sys.getrefcount` reports one extra reference for its own argument. The helper in the package removes it.

In [ ]:
from understanding_python.internals import refcount, Tracked, make_cycle, collect
x = object()
base = refcount(x)
y = x
lst = [x]
assert refcount(x) == base + 2
del y, lst
assert refcount(x) == base

Tracked.reclaimed.clear()
t = Tracked("solo")
del t                       # count hits zero, __del__ runs immediately
assert Tracked.reclaimed == ["solo"]

## Cycles and the garbage collector
Two objects that reference each other never reach zero. The cyclic collector finds unreachable cycles by tracing from roots.

In [ ]:
import gc
Tracked.reclaimed.clear()
make_cycle("loop")
assert Tracked.reclaimed == []          # refcounts stay above zero
found = collect()
assert found >= 2 and set(Tracked.reclaimed) == {"loop-a", "loop-b"}
print("generation thresholds:", gc.get_threshold())

## Interning and the small-integer cache
CPython keeps one object for each integer from -5 to 256. Identifier-like string constants are interned.
`sys.intern` forces interning. Never rely on `is` for values; this cell shows why.

In [ ]:
import sys
a, b = int("256"), int("256")
c, d = int("100000"), int("100000")
assert a is b                 # cached small integers
assert c == d and c is not d  # large integers are separate objects
s1 = "".join(["py", "thon", "!"])
s2 = "".join(["py", "thon", "!"])
assert s1 == s2 and s1 is not s2
assert sys.intern(s1) is sys.intern(s2)

## Object sizes
`sys.getsizeof` reports the shallow size. A list over-allocates, so appends are amortised O(1).

**Proof sketch.** Grow capacity by a factor g > 1. Each resize copies the current n items. Over m appends the copies sum to a geometric series
bounded by m * g / (g - 1), so the total work is O(m) and each append costs O(1) amortised.

In [ ]:
from understanding_python.internals import deep_size
sizes = []
lst = []
for i in range(64):
    lst.append(i)
    sizes.append(sys.getsizeof(lst))
jumps = sum(1 for p, q in zip(sizes, sizes[1:]) if q != p)
print("distinct resizes in 64 appends:", jumps)
assert jumps < 20 and sizes == sorted(sizes)
assert sys.getsizeof(()) < sys.getsizeof([])
assert deep_size({"k": [1, 2, 3]}) > sys.getsizeof({"k": [1, 2, 3]})

## Bytecode with `dis`
The compiler turns a function into bytecode for a stack machine. Reading it shows what a line really does.

In [ ]:
from understanding_python.internals import opnames, disassemble
def add_scaled(a, b):
    return a + 2 * b
ops = opnames(add_scaled)
print(disassemble(add_scaled))
assert any(o.startswith("LOAD") for o in ops)
assert any(o.startswith("RETURN") for o in ops)
assert add_scaled.__code__.co_varnames[:2] == ("a", "b")

## Pitfall: mutable default arguments
Python evaluates a default once, when the function is defined. All calls share that object.

In [ ]:
def bad(item, bucket=[]):
    bucket.append(item)
    return bucket
def good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket
assert bad(1) == [1] and bad(2) == [1, 2]
assert good(1) == [1] and good(2) == [2]
import copy
nested = [[1], [2]]
shallow, deep = copy.copy(nested), copy.deepcopy(nested)
nested[0].append(9)
assert shallow[0] == [1, 9] and deep[0] == [1]

## Benchmark: attribute storage
A class with `__slots__` stores attributes in fixed slots and skips the per-instance dict.

In [ ]:
from understanding_python.bench import best_time
class Plain:
    def __init__(self): self.a, self.b = 1, 2
class Slotted:
    __slots__ = ("a", "b")
    def __init__(self): self.a, self.b = 1, 2
p, s = Plain(), Slotted()
assert hasattr(p, "__dict__") and not hasattr(s, "__dict__")
print("plain", sys.getsizeof(p) + sys.getsizeof(p.__dict__), "slotted", sys.getsizeof(s))
assert sys.getsizeof(s) < sys.getsizeof(p) + sys.getsizeof(p.__dict__)
print(f"create plain {best_time(Plain, 20000):.4f}s slotted {best_time(Slotted, 20000):.4f}s")